In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval[ontology]
except Exception:
    pass

In [2]:
from dataeval import Ontology
from dataeval.core import label_reconciliation
from dataeval.types import OntologyConcept

In [3]:
ontology = Ontology.from_hierarchy({
    "vehicle": {
        "land vehicle": {"sedan": None, "pickup truck": None},
        "watercraft": {"frigate": None, "cargo ship": None},
        "aircraft": {"airliner": None, "fighter jet": None},
    },
})
print(ontology)

Ontology(10 concepts, 1 roots, 6 leaves, 0 external)


In [4]:
index2label = {0: "sedan", 1: "pickup truck", 2: "fighter jet", 3: "rowboat"}

result = label_reconciliation(index2label.values(), ontology)

In [5]:
print("keys:", list(result))

keys: ['matched', 'unmatched', 'ambiguous', 'ancestor_paths', 'external_ancestors', 'induced_edges', 'relations']


In [6]:
print("matched:  ", result["matched"])
print("unmatched:", result["unmatched"])
print("ambiguous:", result["ambiguous"])

matched:   {'sedan': 'sedan', 'pickup truck': 'pickup truck', 'fighter jet': 'fighter jet'}
unmatched: ['rowboat']
ambiguous: {}


In [7]:
# Each matched class's is-a path, from nearest parent up to the root
for name, path in result["ancestor_paths"].items():
    print(f"{name:>14}  <-  {' < '.join(path)}")

         sedan  <-  land vehicle < vehicle
  pickup truck  <-  land vehicle < vehicle
   fighter jet  <-  aircraft < vehicle


In [8]:
print("sedan vs pickup truck:", result["relations"][("sedan", "pickup truck")])
print("sedan vs fighter jet: ", result["relations"][("sedan", "fighter jet")])

sedan vs pickup truck: sibling
sedan vs fighter jet:  sibling


In [9]:
label_reconciliation(["vehicle", "land vehicle", "sedan"], ontology)["induced_edges"]

[('vehicle', 'land vehicle'), ('land vehicle', 'sedan')]

In [10]:
JSONLD = """
{
  "@context": {
    "owl": "http://www.w3.org/2002/07/owl#",
    "subClassOf": {"@id": "http://www.w3.org/2000/01/rdf-schema#subClassOf", "@type": "@id"},
    "prefLabel": {"@id": "http://www.w3.org/2004/02/skos/core#prefLabel"},
    "altLabel": {"@id": "http://www.w3.org/2004/02/skos/core#altLabel"},
    "definition": {"@id": "http://www.w3.org/2004/02/skos/core#definition"},
    "cv": "http://example.org/cv#"
  },
  "@graph": [
    {"@id": "cv:Aircraft", "@type": "owl:Class", "prefLabel": "Aircraft"},
    {"@id": "cv:FighterJet", "@type": "owl:Class", "subClassOf": "cv:Aircraft",
     "prefLabel": "Fighter Jet",
     "altLabel": ["F-16", "Viper"],
     "definition": "A fast, maneuverable military aircraft."}
  ]
}
"""

owl_ontology = Ontology.from_rdf(JSONLD, format="json-ld")
print(owl_ontology)

Ontology(2 concepts, 1 roots, 1 leaves, 0 external)


In [11]:
print("find('F-16'):", owl_ontology.find("F-16"))

concept = owl_ontology.concept("http://example.org/cv#FighterJet")
print("label:     ", concept.label)
print("synonyms:  ", concept.synonyms)
print("definition:", concept.definition)

find('F-16'): ('http://example.org/cv#FighterJet',)
label:      Fighter Jet
synonyms:   ('F-16', 'Viper')
definition: A fast, maneuverable military aircraft.


In [12]:
subset = Ontology([
    # 'warship' is referenced as a parent but never defined in this subset
    OntologyConcept(id="frigate", label="Frigate", parents=("warship",)),
    OntologyConcept(id="sedan", label="Sedan"),
])
print("external_ids:", subset.external_ids)

subset_result = label_reconciliation(["Frigate", "Sedan"], subset)
print("external_ancestors:", subset_result["external_ancestors"])

external_ids: ('warship',)
external_ancestors: {'Frigate': ['warship']}


In [13]:
print("ancestors(sedan): ", ontology.ancestors("sedan"))
print("siblings(sedan):  ", ontology.siblings("sedan"))
print("descendants(vehicle):", ontology.descendants("vehicle"))
print("depth_of(sedan):  ", ontology.depth_of("sedan"))
print("leaves:           ", ontology.leaves)

ancestors(sedan):  ('land vehicle', 'vehicle')
siblings(sedan):   ('pickup truck',)
descendants(vehicle): ('land vehicle', 'watercraft', 'aircraft', 'sedan', 'pickup truck', 'frigate', 'cargo ship', 'airliner', 'fighter jet')
depth_of(sedan):   2
leaves:            ('sedan', 'pickup truck', 'frigate', 'cargo ship', 'airliner', 'fighter jet')


In [14]:
watercraft = ontology.subtree("watercraft")
print(repr(watercraft))
print("ids:", sorted(watercraft.ids))

Ontology(3 concepts, 1 roots, 2 leaves, 0 external)
ids: ['cargo ship', 'frigate', 'watercraft']
